In [ ]:
query = """WITH startTimes AS (
    SELECT 
        gameUID
        , localDateTime
    FROM [base].[mitts_trackman_play]
    WHERE pitchNo = 1
    GROUP BY gameUID, localDateTime
),
endTimes AS (
    SELECT 
        gameUID
        , MAX(localDateTime) AS localDateTime
    FROM [base].[mitts_trackman_play]
    GROUP BY gameUID
)
SELECT
    PLAY.playID
    , PLAY.gameUID
    , PLAY.localDateTime
    , startTimes.localDateTime AS startTime
    , endTimes.localDateTime AS endTime
    , CASE WHEN DATEPART(HOUR, PLAY.localDateTime) < 5 THEN DATEADD(DAY, -1, PLAY.localDateTime) ELSE PLAY.localDateTime END AS dateNew
    , EBE.date
    , PLAY.batterTeam
    , PLAY.pitcherTeam
    , pitchCall
    , playResult
    , balls
    , strikes
    , outs
    , plateLocHeight
    , plateLocSide
    , in_zone
    , zone_loc
    , hit_type_classification
    , in_play
    , r_1B
    , r_2B
    , r_3B
    , PLAY.inning
    , PLAY.PAofInning
    , EBE.PAofInning
    , PLAY.topBottom
    , outsOnPlay
    , runsScored
    , batterName
    , pitcherName
    , pitchofPA
    , max_event_of_PA
    , event_of_PA
    , r1_name
    , r2_name
    , r3_name
    , description
    , SESS.leagueShortName
    
    
FROM [base].[mitts_trackman_play] PLAY
JOIN [base].[mitts_trackman_pitch] PITCH ON PITCH.playID = PLAY.playID
JOIN [base].[mitts_trackman_session] SESS ON SESS.gameUID = PLAY.gameUID
JOIN [base].[vw_mitts_pbp_outcomes] pbp ON pbp.playID = PLAY.playID
-- Get the baserunner state
--JOIN [base].[baserunner_state] BR ON BR.playID = PLAY.playID
JOIN startTimes ON startTimes.gameUID = PLAY.gameUID 
JOIN endTimes ON endTimes.gameUID = PLAY.gameUID
JOIN [base].[scraped_baserunner_ebe] EBE ON PLAY.batterTeam = EBE.batterTeam AND PLAY.pitcherTeam = EBE.pitcherTeam AND PLAY.inning = EBE.inning
                AND PLAY.PAofInning = EBE.PAofInning AND PLAY.topBottom = EBE.topBottom AND YEAR(EBE.date) = YEAR(PLAY.localDateTime) AND MONTH(EBE.date) = MONTH(PLAY.localDateTime)
                AND EBE.date = CASE 
                    WHEN CAST(PLAY.localDateTime AS DATE) != CAST(startTimes.localDateTime AS DATE) 
                    THEN CAST(startTimes.localDateTime AS DATE)
                    ELSE CAST(PLAY.localDateTime AS DATE) 
                END
             
WHERE  ((pitchofPA > 1 AND event_of_PA = max_event_of_PA) OR (pitchofPA = 1 AND event_of_PA = 1))
AND (YEAR(PLAY.localDateTime) IN (2023,2024, 2025))  
AND (PLAY.batterTeam IN (SELECT teamName FROM [base].[vw_mitts_power_5_plus]) AND PLAY.pitcherTeam IN (SELECT teamName FROM [base].[vw_mitts_power_5_plus]))
AND (PLAY.localDateTime >= startTimes.localDateTime) AND (PLAY.localDateTime <= endTimes.localDateTime)
ORDER BY PLAY.gameUID, PLAY.localDateTime"""

In [ ]:

import pandas as pd

In [ ]:
#Password hidden from public

In [ ]:
data = db_conn.run_query(query, file=False)
data


In [ ]:
print(data.columns)
data['playResult'].value_counts()



In [ ]:
# Define strikes
data["is_strike"] = data["pitchCall"].isin(["strikecalled", "strikeswinging", "foulballnotfieldable"])

# Define in-zone pitches
data["is_inzone"] = data["in_zone"] == 1.0


In [ ]:
# Define leverage situation
leverage = ((data["r_2B"] == True) | (data["r_3B"] == True)) & (data["outs"] < 2)
data_leverage = data[leverage]

# Overall strike %
overall_strike = (data.groupby(["pitcherName", "pitcherTeam"])
    .agg(total_pitches=("pitchCall", "size"),
         total_strikes=("is_strike", "sum"))
    .assign(strike_percentage=lambda x: (x["total_strikes"] / x["total_pitches"]) * 100))

# Leverage strike %
leverage_strike = (data_leverage.groupby(["pitcherName", "pitcherTeam"])
    .agg(leverage_pitches=("pitchCall", "size"),
         leverage_strikes=("is_strike", "sum"))
    .assign(leverage_strike_percentage=lambda x: (x["leverage_strikes"] / x["leverage_pitches"]) * 100))

# Merge strike stats and reset index
strike_comparison = overall_strike.join(leverage_strike, how="left").fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
strike_comparison = strike_comparison.reset_index()

# Apply filters
strike_comparison = strike_comparison.query('total_pitches > 200 and leverage_pitches > 30')

# Display final table
strike_comparison

In [ ]:
# Define leverage situation (if not already defined)
leverage = ((data["r_2B"] == True) | (data["r_3B"] == True)) & (data["outs"] < 2)
data_leverage = data[leverage]

# Overall in-zone %
overall_inzone = (data.groupby(["pitcherName", "pitcherTeam"])
    .agg(total_pitches=("pitchCall", "size"),
         overall_inzone_count=("is_inzone", "sum"))
    .assign(inzone_percentage=lambda x: (x["overall_inzone_count"] / x["total_pitches"]) * 100))

# Leverage in-zone %
leverage_inzone = (data_leverage.groupby(["pitcherName", "pitcherTeam"])
    .agg(leverage_pitches=("pitchCall", "size"),
         leverage_inzone_count=("is_inzone", "sum"))
    .assign(leverage_inzone_percentage=lambda x: (x["leverage_inzone_count"] / x["leverage_pitches"]) * 100))

# Merge in-zone stats and reset index
inzone_comparison = overall_inzone.join(leverage_inzone, how="left").fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
inzone_comparison = inzone_comparison.reset_index()

# Apply filters
inzone_comparison = inzone_comparison.query('total_pitches > 200 and leverage_pitches > 30')

# Display final table
inzone_comparison

In [ ]:
babip_overall = (data.groupby(['pitcherName', 'pitcherTeam'])
    .agg(PitchCount=('playResult', 'size'),
         hit=('playResult', lambda x: x.isin(['single', 'double', 'triple']).sum()),
         BIP=('playResult', lambda x: (~x.isin(['homerun', 'undefined', 'caughtstealing', 'stolenbase'])).sum()))
    .assign(BABIP=lambda df: df['hit'] / df['BIP']))

# Define leverage situations (runner on 2nd or 3rd and <2 outs)
leverage = ((data['r_2B'] == True) | (data['r_3B'] == True)) & (data['outs'] < 2)
data_leverage = data[leverage]

# Leverage BABIP
babip_leverage = (data_leverage.groupby(['pitcherName', 'pitcherTeam'])
    .agg(leverage_pitches=('playResult', 'size'),
         leverage_hit=('playResult', lambda x: x.isin(['single', 'double', 'triple']).sum()),
         leverage_BIP=('playResult', lambda x: (~x.isin(['homerun', 'undefined', 'caughtstealing', 'stolenbase'])).sum()))
    .assign(Leverage_BABIP=lambda df: df['leverage_hit'] / df['leverage_BIP']))

# Merge and reset index
babip_comparison = babip_overall.join(babip_leverage, how='left').fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
babip_comparison = babip_comparison.reset_index()

# Apply minimum pitch filters
babip_comparison = babip_comparison.query('PitchCount > 200 and leverage_pitches > 30 and leverage_hit > 5')

# Display final table
babip_comparison

In [ ]:
walk = (data['balls'] == 3) & (data['pitchCall'] == 'ballcalled')

# Identify PA-ending results (excluding walks)
pa_results = ['out', 'single', 'double', 'triple', 'homerun',
              'fielderschoice', 'sacrifice', 'error']

# Tag all pitches for convenience
data = data.copy()
data['is_walk'] = walk
data['is_pa'] = data['playResult'].isin(pa_results) | data['is_walk']

# Overall Walk Stats
overall_walk = (data.groupby(['pitcherName', 'pitcherTeam'])
    .agg(PitchCount=('pitchCall', 'size'),
         PA=('is_pa', 'sum'),         # total plate appearances
         BB=('is_walk', 'sum'))       # walks
    .assign(Walk_Percentage=lambda x: (x['BB'] / x['PA']) * 100))

# Define leverage situations
leverage = ((data['r_2B'] == True) | (data['r_3B'] == True)) & (data['outs'] < 2)
data_leverage = data[leverage]

# Leverage Walk Stats
leverage_walk = (data_leverage.groupby(['pitcherName', 'pitcherTeam'])
    .agg(leverage_pitches=('pitchCall', 'size'),
         leverage_PA=('is_pa', 'sum'),
         leverage_BB=('is_walk', 'sum'))
    .assign(Leverage_Walk_Percentage=lambda x: (x['leverage_BB'] / x['leverage_PA']) * 100))

# Merge and reset index
walk_comparison = overall_walk.join(leverage_walk, how='left').fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
walk_comparison = walk_comparison.reset_index()

# Apply minimum pitch filters
walk_comparison = walk_comparison.query('PitchCount > 200 and leverage_pitches > 30')

# Display final table
walk_comparison

In [ ]:
# Chase swing = pitch in "chase" zone that was swung at (not taken)
swing_calls = ['strikeswinging', 'foulballnotfieldable', 'inplay']

# Overall Chase %
overall_chase = (data.groupby(['pitcherName', 'pitcherTeam'])
    .agg(PitchCount=('pitchCall', 'size'),
         chase_swings=('pitchCall', 
                       lambda x: ((data.loc[x.index, 'zone_loc'] == 'chase') & 
                                  (x.isin(swing_calls))).sum()),
         total_chase=('zone_loc', lambda x: (x == 'chase').sum()))
    .assign(Chase_Percentage=lambda x: (x['chase_swings'] / x['total_chase']) * 100))

# Leverage Chase %
leverage = ((data['r_2B'] == True) | (data['r_3B'] == True)) & (data['outs'] < 2)
data_leverage = data[leverage]

leverage_chase = (data_leverage.groupby(['pitcherName', 'pitcherTeam'])
    .agg(leverage_pitches=('pitchCall', 'size'),
         leverage_chase_swings=('pitchCall',
                                lambda x: ((data_leverage.loc[x.index, 'zone_loc'] == 'chase') & 
                                           (x.isin(swing_calls))).sum()),
         leverage_total_chase=('zone_loc', lambda x: (x == 'chase').sum()))
    .assign(Leverage_Chase_Percentage=lambda x: (x['leverage_chase_swings'] / x['leverage_total_chase']) * 100))

# Merge and reset index
chase_comparison = overall_chase.join(leverage_chase, how='left').fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
chase_comparison = chase_comparison.reset_index()

# Apply filters
chase_comparison = chase_comparison.query('PitchCount > 200 and leverage_pitches > 30')
# Display final table
chase_comparison

In [ ]:
# Overall Heart %
heart_overall = (data.groupby(['pitcherName', 'pitcherTeam'])
    .agg(PitchCount=('pitchCall', 'size'),
         heart_pitches=('zone_loc', lambda x: (x == 'heart').sum()))
    .assign(Heart_Percentage=lambda df: (df['heart_pitches'] / df['PitchCount']) * 100))

# Define leverage situations (runner on 2nd or 3rd and <2 outs)
leverage = ((data['r_2B'] == True) | (data['r_3B'] == True)) & (data['outs'] < 2)
data_leverage = data[leverage]

# Leverage Heart %
heart_leverage = (data_leverage.groupby(['pitcherName', 'pitcherTeam'])
    .agg(leverage_pitches=('pitchCall', 'size'),
         leverage_heart_pitches=('zone_loc', lambda x: (x == 'heart').sum()))
    .assign(Leverage_Heart_Percentage=lambda df: (df['leverage_heart_pitches'] / df['leverage_pitches']) * 100))

# Merge Overall + Leverage Heart %
heart_comparison = heart_overall.join(heart_leverage, how='left').fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
heart_comparison = heart_comparison.reset_index()

# Apply minimum pitch filters
heart_comparison = heart_comparison.query('PitchCount > 200 and leverage_pitches > 30')

# Display final table
heart_comparison

In [ ]:
#Define leverage situation
leverage = ((data["r_2B"] == True) | (data["r_3B"] == True)) & (data["outs"] < 2)
data_leverage = data[leverage].copy()

#Define Ideal Contact types
ideal_contact_types = ["Flare-or-Burner", "Solid-Contact", "Barrel"]

#Create is_ideal_contact for ALL pitches
data["is_ideal_contact"] = None
data.loc[(data["in_play"] == 1) & (data["hit_type_classification"].isin(ideal_contact_types)), "is_ideal_contact"] = 1.0
data.loc[(data["in_play"] == 1) & (~data["hit_type_classification"].isin(ideal_contact_types)), "is_ideal_contact"] = 0.0

# --- Same for leverage subset ---
data_leverage["is_ideal_contact"] = None
data_leverage.loc[(data_leverage["in_play"] == 1) & (data_leverage["hit_type_classification"].isin(ideal_contact_types)), "is_ideal_contact"] = 1.0
data_leverage.loc[(data_leverage["in_play"] == 1) & (~data_leverage["hit_type_classification"].isin(ideal_contact_types)), "is_ideal_contact"] = 0.0

# --- Overall ICR
overall_icr = (data.groupby(["pitcherName", "pitcherTeam"])
    .agg(total_pitches=("pitchCall", "size"),
         Ideal_Contact_Rate=("is_ideal_contact", "mean"))
    .assign(Ideal_Contact_Rate=lambda x: x["Ideal_Contact_Rate"] * 100))

#Leverage ICR
leverage_icr = (data_leverage.groupby(["pitcherName", "pitcherTeam"])
    .agg(leverage_pitches=("pitchCall", "size"),
         Leverage_Ideal_Contact_Rate=("is_ideal_contact", "mean"))
    .assign(Leverage_Ideal_Contact_Rate=lambda x: x["Leverage_Ideal_Contact_Rate"] * 100))

#Merge overall and leverage
icr_comparison = overall_icr.join(leverage_icr, how="left").fillna(0)

# Reset index to make pitcherName and pitcherTeam regular columns
icr_comparison = icr_comparison.reset_index()

# Apply pitch count filters
final_icr = icr_comparison[(icr_comparison["total_pitches"] >= 200) & (icr_comparison["leverage_pitches"] >= 30)]
final_icr

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_kpi_delta(df, overall_col, leverage_col, kpi_name, 
                   worse_is_higher=True, top_n=20, team=None, 
                   sort_by='delta', show_sample_size=True, show_league_avg=True,
                   figsize=(12, 10)):
    """Plot delta between overall and leverage performance
    
    Parameters:
    df : DataFrame with pitcher performance data
    overall_col : name of overall KPI column
    leverage_col : name of leverage KPI column
    kpi_name : descriptive name for titles/labels
    worse_is_higher : True if higher values are worse (ICR, Chase%, Heart%, Walk%)
                      False if higher values are better (Strike%, In-Zone%)
    top_n : number of pitchers to display (default 20, None for all)
    team : if specified, filter to only pitchers from this team (e.g., 'CLE_TIG')
    sort_by : 'delta' (default), 'abs_delta', 'overall', or 'leverage'
    show_sample_size : if True, show leverage pitch count next to pitcher names
    show_league_avg : if True, show league average delta line"""
    
    # Make a copy and ensure pitcherName is a column, not index
    df_plot = df.copy()
    if df_plot.index.name == 'pitcherName' or 'pitcherName' not in df_plot.columns:
        df_plot = df_plot.reset_index()
    
    # Filter by team if specified
    if team is not None:
        if 'pitcherTeam' in df_plot.columns:
            df_plot = df_plot[df_plot['pitcherTeam'] == team]
            if len(df_plot) == 0:
                print(f"Warning: No pitchers found for team '{team}'")
                return None, None
        else:
            print(f"Warning: 'pitcherTeam' column not found in DataFrame")
    
    # Calculate delta and absolute delta
    df_plot['delta'] = df_plot[leverage_col] - df_plot[overall_col]
    df_plot['abs_delta'] = df_plot['delta'].abs()
    
    # Calculate league average delta
    league_avg_delta = df_plot['delta'].mean()
    
    # Determine sorting column
    sort_column_map = {
        'delta': 'delta',
        'abs_delta': 'abs_delta',
        'overall': overall_col,
        'leverage': leverage_col}
    sort_column = sort_column_map.get(sort_by, 'delta')
    
    # Filter to top N
    if top_n and len(df_plot) > top_n:
        if sort_by == 'abs_delta':
            # For absolute delta, take the top N by absolute value
            df_plot = df_plot.nlargest(top_n, 'abs_delta')
        elif sort_by in ['overall', 'leverage']:
            # For overall or leverage, take top N by that metric
            df_plot = df_plot.nlargest(top_n, sort_column)
        else:
            # For regular delta, take extremes (top and bottom)
            top_half = top_n // 2
            bottom_half = top_n - top_half
            df_plot = pd.concat([
                df_plot.nlargest(top_half, 'delta'),
                df_plot.nsmallest(bottom_half, 'delta')])
    
    # Sort by the chosen column for display
    df_plot = df_plot.sort_values(sort_column)
    
    # Find leverage pitch count column (different names in different dataframes)
    leverage_pitch_col = None
    for col in ['leverage_pitches', 'leverage_PA', 'leverage_BIP']:
        if col in df_plot.columns:
            leverage_pitch_col = col
            break
    
    # Set pitcherName as index for plotting
    df_plot = df_plot.set_index('pitcherName')
    
    # Color coding based on whether higher is worse
    if worse_is_higher:
        colors = ['red' if x > 0 else 'green' for x in df_plot['delta']]
        struggle_label = 'Struggles in Leverage'
        excel_label = 'Excels in Leverage'
    else:
        colors = ['green' if x > 0 else 'red' for x in df_plot['delta']]
        struggle_label = 'Struggles in Leverage'
        excel_label = 'Excels in Leverage'
    
    # Create plot
    fig, ax = plt.subplots(figsize=figsize)
    
    bars = ax.barh(range(len(df_plot)), df_plot['delta'], 
                    color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    # Formatting - add sample size to y-axis labels if requested
    if show_sample_size and leverage_pitch_col:
        ylabels = [f"{name} (n={int(df_plot.loc[name, leverage_pitch_col])})" 
                   for name in df_plot.index]
    else:
        ylabels = df_plot.index
    
    ax.set_yticks(range(len(df_plot)))
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel(f'{kpi_name} Delta (Leverage - Overall) %', fontsize=11, fontweight='bold')
    
    # Add team and sort info to title
    title = f'{kpi_name}: High Leverage Performance vs Overall'
    if team:
        title += f' - {team}'
    title += f'\n(Positive = {"Worse" if worse_is_higher else "Better"} in Leverage)'
    if sort_by != 'delta':
        sort_labels = {'abs_delta': 'Absolute Delta', 'overall': 'Overall Rate', 'leverage': 'Leverage Rate'}
        title += f' | Sorted by: {sort_labels.get(sort_by, sort_by)}'
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Zero line
    ax.axvline(x=0, color='black', linestyle='-', linewidth=2, label='No Change')
    
    # League average line
    if show_league_avg:
        ax.axvline(x=league_avg_delta, color='blue', linestyle='--', linewidth=1.5, 
                   alpha=0.7, label=f'League Avg ({league_avg_delta:.2f}%)')
    
    # Grid
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    
    # Add value labels on bars
    for i, (idx, row) in enumerate(df_plot.iterrows()):
        value = row['delta']
        x_pos = value + (0.2 if value > 0 else -0.2)
        ha = 'left' if value > 0 else 'right'
        ax.text(x_pos, i, f'{value:.2f}%', 
                va='center', ha=ha, fontsize=8, fontweight='bold')
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', alpha=0.7, label=struggle_label),
        Patch(facecolor='green', alpha=0.7, label=excel_label)]
    if show_league_avg:
        from matplotlib.lines import Line2D
        legend_elements.append(Line2D([0], [0], color='blue', linewidth=1.5, 
                                      linestyle='--', label=f'League Avg ({league_avg_delta:.2f}%)'))
    
    ax.legend(handles=legend_elements, loc='best', fontsize=10)
    
    plt.tight_layout()
    return fig, ax

# Usage examples:
fig1, ax1 = plot_kpi_delta(final_icr, 'Ideal_Contact_Rate', 'Leverage_Ideal_Contact_Rate',
                            'Ideal Contact Rate', worse_is_higher=True, 
                            team='CLE_TIG', top_n=20, sort_by='delta', 
                            show_sample_size=True, show_league_avg=True)
plt.show()

fig2, ax2 = plot_kpi_delta(chase_comparison, 'Chase_Percentage', 'Leverage_Chase_Percentage',
                            'Chase Rate', worse_is_higher=False, 
                            team='CLE_TIG', top_n=20, sort_by='delta',
                            show_sample_size=True, show_league_avg=True)
plt.show()

fig3, ax3 = plot_kpi_delta(heart_comparison, 'Heart_Percentage', 'Leverage_Heart_Percentage',
                            'Heart%', worse_is_higher=True, 
                            team='CLE_TIG', top_n=20, sort_by='delta',
                            show_sample_size=True, show_league_avg=True)
plt.show()

fig4, ax4 = plot_kpi_delta(walk_comparison, 'Walk_Percentage', 'Leverage_Walk_Percentage',
                            'Walk Rate', worse_is_higher=True, 
                            team='CLE_TIG', top_n=20, sort_by='delta',
                            show_sample_size=True, show_league_avg=True)
plt.show()

fig5, ax5 = plot_kpi_delta(strike_comparison, 'strike_percentage', 'leverage_strike_percentage',
                            'Strike%', worse_is_higher=False, 
                            team='CLE_TIG', top_n=20, sort_by='delta',
                            show_sample_size=True, show_league_avg=True)
plt.show()

fig6, ax6 = plot_kpi_delta(inzone_comparison, 'inzone_percentage', 'leverage_inzone_percentage',
                            'In-Zone%', worse_is_higher=True, 
                            team='CLE_TIG', top_n=20, sort_by='delta',
                            show_sample_size=True, show_league_avg=True)
plt.show()

fig7, ax7 = plot_kpi_delta(babip_comparison, 'BABIP', 'Leverage_BABIP',
                            'BABIP', worse_is_higher=True, 
                            team='CLE_TIG', top_n=20, sort_by='delta',
                            show_sample_size=True, show_league_avg=True)
plt.show()